# MnesOS Orchestrator Showcase

This notebook demonstrates how to use the `Orchestrator` for an interactive RPG session. It implements a **Middle Out** context management strategy and persistent logging.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 1. Ensure CWD is at repository root for load_dotenv and relative paths
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"Project Root: {os.getcwd()}")

# 2. Ensure the source is in path
sys.path.append(os.path.abspath("src"))

from MnesOS.orchestrator import Orchestrator
from langchain_openai import ChatOpenAI

load_dotenv() # Now correctly looks in repo root for .env

## 1. Initialization

We initialize the LLM and the Orchestrator with a cartridge.

In [ ]:
llm = ChatOpenAI(model="google/gemini-2.5-flash", temperature=0.9)

orch = Orchestrator(
    cartridge_dir="cartridges/generic_rpg",
    llm_director=llm,
    llm_npc=llm,
    llm_narrator=llm
)

print("Game Ready!")

## 2. Config & Helpers

Define the Middle Out logic and logging configuration.

In [ ]:
MAX_HISTORY = 10
FRONT_KEPT = 2
LOG_FILE = "data/game-play.md"

def ensure_log_initialized():
    if not os.path.exists(LOG_FILE):
        Path(LOG_FILE).parent.mkdir(parents=True, exist_ok=True)
        with open(LOG_FILE, "w") as f:
            f.write("# MnesOS Game Play Session\n\n")

def middle_out_truncate(messages, max_len=MAX_HISTORY, front_len=FRONT_KEPT):
    if len(messages) <= max_len:
        return messages
    back_len = max_len - front_len
    return messages[:front_len] + messages[-back_len:]

def log_turn(user_input, response):
    with open(LOG_FILE, "a") as f:
        f.write(f"### Player\n{user_input}\n\n")
        f.write(f"### Narrator\n{response}\n\n---\n\n")

In [ ]:
print("RPG Started. Type 'exit' to stop.\n")
ensure_log_initialized()

# Load first-message.md if it exists in the cartridge
first_msg_path = Path(orch.cartridge.lore_path).parent / "first-message.md"
if first_msg_path.exists():
    first_msg_content = first_msg_path.read_text()
    print(f"NARRATOR: {first_msg_content}\n")
    # Initialize client_messages with the first message if it's empty
    if not orch._state['client_messages']:
        orch._state['client_messages'].append({"role": "assistant", "content": first_msg_content})

## 3. Interactive Loop

Run this cell to play the game indefinitely. Type `exit` or `quit` to stop.

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit", "q"]:
        break
    
    # 1. Print User Input (so it's visible in console history)
    print(f"\n> PLAYER: {user_input}")
    
    # 2. Process Turn
    response = orch.process_turn(user_input)
    
    # 3. Display Response
    print(f"\nNARRATOR: {response}\n")
    
    # 4. Persistent Logging (Append Only)
    log_turn(user_input, response)
    
    # 5. Middle Out Truncation (Context Management)
    orch._state['client_messages'] = middle_out_truncate(orch._state['client_messages'])
    
    # print(f"[Context Size: {len(orch._state['client_messages'])} messages]")